# D1.7 · Drift monitoring

**Function D — Security Operations → The SOC Analyst & Detection Engineer**  ·  *Security of AI*

---

**Risk.** A detection that worked last month is silently degraded.

**Control.** Watch model updates, prompt changes, index refreshes, tool versions.

**This lab.** Catch a detection silently degrading after a model change.

| | |
|---|---|
| Open-source tooling | promptfoo |
| Open-weight models | GLM-4.6 |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("D1.7"))

Drift monitoring exists because an agent's behaviour changes without a code change. The control you signed off was tested against a behaviour that no longer exists.

In [ ]:
from cybercommons import soc, grc
import time

now = time.time()
base = soc.Baseline(tool_mix={"read_file": 0.7, "search_code": 0.2, "write_file": 0.1},
                    actions_per_hour=300)

WEEKS = {
 "week 1 (baseline)":  {"read_file": 70, "search_code": 20, "write_file": 10},
 "week 4 (new prompt)": {"read_file": 55, "search_code": 20, "write_file": 25},
 "week 8 (model upgrade)": {"read_file": 30, "search_code": 10, "write_file": 25,
                            "run_shell": 35},
}
for label, mix in WEEKS.items():
    ev = [soc.Event(now, "agent", tool) for tool, n in mix.items() for _ in range(n)]
    d = base.compare(ev)
    print(f"{label:26s} drift={d['drift']:.3f}  {d['verdict']}")
    if d["new_tools"]:
        print(f"{'':26s} new tools: {d['new_tools']}")

Now connect it to the control that was signed off against week 1.

In [ ]:
tests = [grc.ControlTest("SB-1", True, "week-1 egress test",
                         tested_at=now - 56 * 86400)]
print(grc.verify_continuously(tests, ["SB-1", "DR-1"], now=now))

The control passed — eight weeks ago, against a tool mix that has since changed by more than half. `STALE` is the honest state, and it is the one point-in-time testing cannot report.

### Expect

Drift rises across the three weeks, week 8 flags `run_shell` as a new tool and reports significant drift, and the eight-week-old control test is reported STALE with coverage 0.0.

### Your turn

Set the freshness window for one of your agent controls. Justify the number from your observed drift rate rather than from the audit calendar.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/D1.7.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*